<a href="https://colab.research.google.com/github/zatkins2/DR6_Notebooks/blob/main/ACT_DR6_depth1_maps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading and Analyzing Depth-1 Maps

*Written by Allen Foster and the ACT collaboration*

---

Depth-1 maps are the collection of observations into maps that contain information only from one rotation of the sky through the focal plane. Depending on the scan strategy, the set of observations making up one depth-1 map may vary and therefore so does the extent, noise properties, and sky location of these maps.

Each depth-1 mapset can be thought of as one observation of the sky. Each mapset contains an intensity and inverse-variance map as well as a detector hits-weighted time map.

A matched filtered version of these maps is also available as rho (inverse-variance weighted intensity) and kappa (inverse variance) maps.

All maps are saved in FITS file format, with the exception of a metadata file with suffix `info.hdf` which contains metadata from the observation(s), most notably the starting time.

---

The purpose of this notebook is two-fold:

The first is to describe the depth-1 mapset, how to load the matched filtered maps, extract sources using the photutils package, match catalogs, and search for transients.


The second, and perhaps more important, is to help understand the unique noise properties, scan patterns and common pitfalls when dealing with these depth-1 maps.

---

If you are not familiar with ACT matched filtering or reading in fits maps using pixell, see [the pixell tutorial](https://github.com/simonsobs/pixell_tutorials/blob/master/Pixell_matched_filtering.ipynb) on matched filtering, which can easily be opened in colab.

The depth1 map used in this example contains a bright stellar flare (source ID J192832-3507.9) from the paper *The Atacama Cosmology Telescope: Systematic Transient Search of Single Observation Maps* -- https://arxiv.org/abs/2409.08429

**Required non-standard Python packages:**

    pixell
    astropy
    photutils

**Required files:**

    PS_S19_f090_2pass_optimalCatalog.fits
    mask_for_sources2019_plus_dust.fits
    depth1_map_tools.py
    source_tools.py
    tiles.py


Grab the necessary software. We also need to manually download the custom modules so colab can locate them in the system paths.

In [ ]:
%%capture
!pip install pixell photutils tqdm matplotlib

In [ ]:
%%capture
!wget https://raw.githubusercontent.com/zatkins2/DR6_Notebooks/refs/heads/main/depth1_maps_modules/depth1_map_tools.py
!wget https://raw.githubusercontent.com/zatkins2/DR6_Notebooks/refs/heads/main/depth1_maps_modules/source_tools.py
!wget https://raw.githubusercontent.com/zatkins2/DR6_Notebooks/refs/heads/main/depth1_maps_modules/tiles.py

Downloading all data products may take ~5 min!

In [ ]:
%%capture
%env USER=abc
%env PASSWORD=123
!wget --user $USER --password $PASSWORD https://phy-act1.princeton.edu/private/data/dr6_depth1_v1/act_sample_cat.fits
!wget --user $USER --password $PASSWORD https://phy-act1.princeton.edu/private/data/dr6_depth1_v1/mask_for_sources2019_plus_dust.fits
!wget --user $USER --password $PASSWORD https://phy-act1.princeton.edu/private/data/dr6_depth1_v1/maps/15386/depth1_1538613353_pa5_f150_info.hdf
!wget --user $USER --password $PASSWORD https://phy-act1.princeton.edu/private/data/dr6_depth1_v1/maps/15386/depth1_1538613353_pa5_f150_ivar.fits
!wget --user $USER --password $PASSWORD https://phy-act1.princeton.edu/private/data/dr6_depth1_v1/maps/15386/depth1_1538613353_pa5_f150_kappa.fits
!wget --user $USER --password $PASSWORD https://phy-act1.princeton.edu/private/data/dr6_depth1_v1/maps/15386/depth1_1538613353_pa5_f150_map.fits
!wget --user $USER --password $PASSWORD https://phy-act1.princeton.edu/private/data/dr6_depth1_v1/maps/15386/depth1_1538613353_pa5_f150_rho.fits
!wget --user $USER --password $PASSWORD https://phy-act1.princeton.edu/private/data/dr6_depth1_v1/maps/15386/depth1_1538613353_pa5_f150_time.fits

In [ ]:
import numpy as np
from glob import glob
from matplotlib import pylab as plt
import warnings

## because many sky maps contain regions which are unobserved, there are zero weights
## let's ignore the divide by zero warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

from pixell import enmap, enplot
from pixell import utils as pixell_utils

In [ ]:
## used for masking the maps
galactic_dust_mask_file = 'mask_for_sources2019_plus_dust.fits'

## where the depth-1 maps are located
map_directory = './'
rho_maps = glob(map_directory + '*rho.fits')

Depth1 mapsets are split by detector array and by frequency band. Here we can see all the depth1 files for a given array and band of a given observation.

In [ ]:
## This function takes in a depth-1 mapset rho map, and extracts the observation ID, detector array and frequency.
## It then uses those to glob all the files for that depth-1 mapset.
def get_relevent_files(rho_map_file, verbose=False):
    obsinfo = rho_map_file.split('/')[-1].split('depth1_')[1].split('_rho.fits')[0]
    if verbose:
        print(obsinfo)

    relevent_files = glob(map_directory+f'*{obsinfo}*')

    if verbose:
        for rf in relevent_files:
            print(rf)
    return relevent_files

In [ ]:
relevent_files = get_relevent_files(rho_maps[0], verbose=True)

**Inside the info.hdf file are several useful pieces of information, including:**

**t**: the start ctime (to be added to the time map to get the actual time each pixel was hit.

**profile**: A curve in RA, dec giving the scanning profile in the map. Equispaced in azimuth. [{dec,ra},nsamp], in radians. Useful for building noise models, filters and masks that handle the curvy nature of our scans.

**array**: The name of the detector array, e.g. pa5_f090.

**ids**: The ids of the TODs used. NB: This includes tods that were were skipped due to missing metadata etc!

**period**: The [ctime_min, ctime_max] defining this period.

**box**: The approximate bounding box of the maps.

**pid**: Which of the 8442 periods we are, counting from 0.

In [ ]:
from pixell.bunch import read as bunch_read

for f in relevent_files:
    if 'info.hdf' in f:
        info = bunch_read(f)
        print(info.keys())
        break

Next, we will load the matched filtered maps and the map of pixel hit times.

rho is the inverse-variance weighted flux map
kappa is the inverse-variance map


In [ ]:
def load_map(map_path:str, pol:str='I') -> enmap.enmap:
    ## map_path should be /file/path/to/[obsid]_[arr]_[freq]_[maptype].fits
    ## pol is a string I,Q, or U, for loading the polarized maps
    ## time and ivar are not polarized, thus only use selector for others.
    pol2sel={'I':0,
             'Q':1,
             'U':2
            }
    if 'time.fits' in map_path or 'ivar.fits' in map_path:
        m = enmap.read_map(map_path)
    else:
        m = enmap.read_map(map_path, sel=pol2sel[pol])

    return m

In [ ]:
## load in the maps, may take a few seconds to load the maps depending on size.
for rf in relevent_files:
    if 'rho' in rf :
        rho_map = load_map(rf)
    elif 'kappa' in rf:
        kappa_map = load_map(rf)
    elif 'time' in rf:
        ## time (in s) since the start of the observation
        time_map = load_map(rf)
    elif 'info' in rf:
        ## also load the starting time of the observation.
        ## this gets added to the time map in order to get absolute time (ctime).
        t0 = bunch_read(rf).t


The flux map can be gotten from the ratio of rho/kappa (and will have units of mJy).

Below, we plot the full flux map from -100 to 100 mJy. If you enlarge the figure, some features become obvious including:
    
    *localized, scan-direction, enhanced noise
    *bright regions in the galactic plane (upper right)
    *bright point sources and masked bright objects (nans because they are masked in the timestreams)
    *enhanced noise along map edges (i.e. where few detectors observe the sky pixel)

In [ ]:
print('Flux map [mJy]')
enplot.pshow(rho_map/kappa_map, range=100, colorbar=True, ticks=15, downgrade=8)

In [ ]:
# plt.figure(figsize=(20,5),dpi=200)
# plt.imshow(rho_map/kappa_map,vmin=-100,vmax=100)
# plt.colorbar(fraction=0.015,pad=0.02,label='Flux (mJy)')
# plt.show()

In [ ]:
## zooming-in on a smaller region near the center of the map
## Sources and regions of increased flux variation become more evident, especially along map edges.
print('Flux map [mJy]')
enplot.pshow((rho_map/kappa_map)[1000:4000, 13000:16000], range=100, colorbar=True, ticks=5, downgrade=4)

In [ ]:
# ## zooming-in on a smaller region near the center of the map
# ## Sources and regions of increased flux variation become more evident, especially along map edges.
# plt.figure(figsize=(10,10),dpi=200)
# plt.imshow((rho_map/kappa_map)[1000:4000,13000:16000],vmin=-100,vmax=100)
# plt.colorbar(fraction=0.04,pad=0.04,label='Flux (mJy)')
# plt.show()

We are also interested not just in the flux, but in the signal to noise ratio (SNR) of sources.
The map noise can be calculated as kappa**(-0.5)

In [ ]:
## Same zoomed-in region as above, but now map noise.
## As you can see the noisy regions of the map occur at the same locations as the enhanced flux above.
## This is likely due to an imperfectly matched field overlap between multiple observations, detector readout problems,
## or short timescale, small angular scale weather events.
print('Noise [mJy]')
enplot.pshow((kappa_map**-0.5)[1000:4000, 13000:16000], min=0, max=70, colorbar=True, ticks=5, downgrade=4, color='viridis')

In [ ]:
# ## Same zoomed-in region as above, but now map noise.
# ## As you can see the noisy regions of the map occur at the same locations as the enhanced flux above.
# ## This is likely due to an imperfectly matched field overlap between multiple observations, detector readout problems,
# ## or short timescale, small angular scale weather events.
# plt.figure(figsize=(10,10),dpi=200)
# plt.imshow((kappa_map**(-0.5))[1000:4000,13000:16000],vmin=0,vmax=50)
# plt.colorbar(fraction=0.04,pad=0.04,label='Noise (mJy)')
# plt.show()

The SNR is then of course flux/noise, or rho / sqrt(kappa). Note SNR can be positive or negative depending on the sign of the flux

In [ ]:
## Same zoomed-in region as above, but now SNR
## A keen eye can see enhanced SNR fluctuations near the edge of the map caused by improper noise modeling
## during matched filtering. We will come back to that during map filtering.
print('SNR')
enplot.pshow((rho_map*kappa_map**-0.5)[1000:4000, 13000:16000], range=5, colorbar=True, ticks=5, downgrade=4)

In [ ]:
# ## Same zoomed-in region as above, but now SNR
# ## A keen eye can see enhanced SNR fluctuations near the edge of the map caused by improper noise modeling
# ## during matched filtering. We will come back to that during map filtering.
# plt.figure(figsize=(10,10),dpi=200)
# plt.imshow((rho_map*kappa_map**(-0.5))[1000:4000,13000:16000],vmin=-5,vmax=5)
# plt.colorbar(fraction=0.04,pad=0.04,label='SNR')
# plt.show()

We can also see when those pixels were hit by looking at the time map of that same zoomed-in region.


The unobserved streak through the middle has a time value of 0.0.

In [ ]:
print('Time since %s [hr]' % t0)
enplot.pshow(time_map[1000:4000, 13000:16000]/(24*60), colorbar=True, ticks=5, downgrade=4, color='viridis')

In [ ]:
# plt.figure(figsize=(5,5),dpi=200)
# plt.imshow(time_map[1000:4000,13000:16000]/(24*60))
# plt.colorbar(fraction=0.04,pad=0.04,label='Time since %s (hr)'%t0)
# plt.show()

In order to deal with some of the localized enhanced noise, further map processing is required.

In [ ]:
from depth1_map_tools import preprocess_map

In [ ]:
## preprocess_map does a few things, notably:
## Apply variance thresholding to the rho and kappa maps in order to remove regions with low weights.
## Cut regions of the map with high variance, using either the median or maximum variance values in the map.
## Mask the galaxy and map edges.

## Flatfielding can be applied, in which case the map is tiled into 1x1 deg chunks
## and the flux and SNR maps are weighted by the noise in their respective chunk; i.e. the filtered noise should be uniform across the map.
## Flatfielding may take a few minutes.

## See the functions in depth1_map_tools for more info.

flux, snr = preprocess_map(rho_map,
                           kappa_map,
                           time_map=time_map,
                           flatfield=True,
                           galmask_file=galactic_dust_mask_file
                           )

In [ ]:
## Plot the flux map, which is now filtered.
## You will notice the galaxy has been masked.
## less obvious are the masked map edges.
print('Flux [mJy]')
enplot.pshow(flux, range=50, colorbar=True, ticks=15, downgrade=8, mask=0)

print('Flux [mJy]')
enplot.pshow(flux[1000:4000, 13000:16000], range=50, colorbar=True, ticks=5, downgrade=4) #, color='viridis')

In [ ]:
## Plot the flux map, which is now filtered.
## You will notice the galaxy has been masked.
## less obvious are the masked map edges.
# plt.figure(figsize=(20,5),dpi=200)
# plt.imshow(flux[1000:4000,13000:16000],vmin=-50,vmax=50)
# plt.colorbar(label='Flux (mJy)',fraction=0.015,pad=0.02)
# plt.show()

Although not as obvious in flux due to the colorbar stretch, it is obvious in SNR that the noise flatfielding helps
to remove the enhanced flucuations near the map edges.

In [ ]:
## Plot the SNR map, which is now filtered.
## You will notice the galaxy has been masked.
## less obvious are the masked map edges.
print('SNR')
enplot.pshow(snr[1000:4000, 13000:16000], range=5, colorbar=True, ticks=5, downgrade=4)

In [ ]:
## Plot the SNR map, which is now filtered.
## You will notice the galaxy has been masked.
## less obvious are the masked map edges.
# plt.figure(figsize=(20,5),dpi=200)
# plt.imshow(snr[1000:4000,13000:16000],vmin=-5,vmax=5)
# plt.colorbar(label='SNR',fraction=0.015,pad=0.02)
# plt.show()

Below, we will load in source finding and catalog matching utilities.

The source finder is built using photutils, and uses the flux map and a map of the local noise (flux/snr) in order to extract peaks above sigma_thresh.
In this example we use a high threshold of 7 sigma due to the modest filtering performed on the map.

Arguments to the source extractor `min_rad` and `sigma_thresh_for_minrad` control the exclusion radius for sources given their SNR; i.e. because bright sources may cause some filtering artefacts and we don't want to "detect" a ton of false positive sources.


In [ ]:
from source_tools import extract_sources, load_act_catalog, crossmatch_sources

In [ ]:
sigma_thresh = 7.0
map_res_arcmin = abs(flux.wcs.wcs.cdelt[0])*pixell_utils.degree/pixell_utils.arcmin
print('Finding sources...')
extracted_sources = extract_sources(flux,
                                    timemap=time_map,
                                    maprms=flux/snr,
                                    nsigma=sigma_thresh,
                                    minrad=[0.5,1.5,3.0,5.0,10.0,20.0,60.0],
                                    sigma_thresh_for_minrad=[0,3,5,10,50,100,200],
                                    res=map_res_arcmin,
                                    )

print(len(extracted_sources.keys()),'sources found.')

In [ ]:
## Load the act source catalog, but this is a sample for the purposes of this notebook
catalog_sources = load_act_catalog()

In [ ]:
# typical_noise = 30.0 # mJy per f090 depth1 map ~1sigma
# flux_thresh_Jy = typical_noise/1000.

# ## Load the act source catalog, but only sources above the typical ~1sigma in the depth-1 map.
# ## This just reduces the need to check for matches with the more numerous dim sources (since we have a large sigma_thresh in the actual source extracting).
# catalog_sources = load_act_catalog(flux_threshold = flux_thresh_Jy)


We will now plot the catalog sources (red circles) and the sources extracted from the depth-1 map (white ellipses).

The map size is quite large (1000x1000 pixels at 0.5 arcminute resolution is ~8x8 degrees)

Although difficult to see at this zoom level, there is one bright source with no red circle ; i.e. no catalog counterpart (near 17700, 3100)! There are also some catalog sources that are not detected in this depth-1 map.

In [ ]:
print('Plotting sources with catalog example...')
## plot SNR map
fig, ax=plt.subplots(figsize=(10, 10), dpi=200)
plt.imshow(snr, vmax=10, vmin=-3)
plt.colorbar()

## get the source catalog and scatter plot them as red circles
ra_zero_cent = catalog_sources['RADeg']
ra_zero_cent[ra_zero_cent>180] -= 360.
y, x = snr.sky2pix([catalog_sources['decDeg']*pixell_utils.degree,
                    ra_zero_cent*pixell_utils.degree])

plt.scatter(x, y, marker='o', s=3, facecolor='none', edgecolor='r', label='Cataloged Sources')

## plot the extracted sources using their phot_utils kron_aperture.
nplotted = 0
for f in range(len(extracted_sources)):
    if not isinstance(extracted_sources[f]['kron_aperture'], type(None)):
        extracted_sources[f]['kron_aperture'].plot(color='w', lw=0.5, label='Extracted Sources' if nplotted==0 else '')
        nplotted += 1
plt.legend()
## just select some smaller region so we can see the sources
plt.xlim([17000, 18000])
plt.ylim([3000, 4000])
plt.show()

Now, we perform the actual crossmatching with the catalog.

By default the minimum crossmatch radius is 1.5 arcminutes, and grows with flux to 30 arcminutes at 1Jy up to a maximum of 2 degrees.

This large crossmatch radius is enforced to reduce false-positives due to filtering artefacts near bright sources.

In [ ]:
print('Cross-matching found sources with catalog...')
source_candidates, transient_candidates, noise_candidates = crossmatch_sources(extracted_sources,
                                                                               catalog_sources,
                                                                               )

In [ ]:
## The output catalogs are lists of dictionaries
## the kron_ output is described on the photutils readthedocs page : https://photutils.readthedocs.io/en/latest/api/photutils.segmentation.SourceCatalog.html#photutils.segmentation.SourceCatalog.kron_flux
print(transient_candidates)

In [ ]:
## simple function to extract a thumbnail map around the ra, dec position.
def get_thumbnail(imap:enmap.ndmap,
                  ra_deg:float,
                  dec_deg:float,
                  size_deg:float=0.5,
                  proj:str='tan',
                  )->enmap:
    from pixell import reproject
    from pixell.utils import degree
    ra = ra_deg * degree
    dec = dec_deg * degree
    ## reproject does do an interpolation in order to reproject to a flat sky map, therefore input needs to be nan_to_num'ed
    omap = reproject.thumbnails(np.nan_to_num(imap),
                                [dec, ra],
                                size_deg * degree,
                                proj=proj,
                                )
    return omap

We can now plot thumbnails around the "transients" to see if they look real.

In [ ]:
print('Transient candidates:')
for tc in transient_candidates:
    thumb = get_thumbnail(snr,
                          tc['ra'],
                          tc['dec'],
                          size_deg=0.5
                          )
    #if np.all(np.isnan(thumb)):
    #    continue

    print(tc['sourceID'])
    print('Flux: %.0f +- %.0f mJy' % (tc['flux'], tc['dflux']))

    plt.imshow(thumb, vmin=-5, vmax=5, cmap='RdYlBu_r')
    plt.title(tc['sourceID'])
    plt.colorbar(label='SNR')
    plt.show()



As an exercise, try reducing the `sigma_thresh` input the source extractor to 6.0.

This will produce two more "transients", although close inspection of the thumbnails will reveal that these are in regions with enhanced noise. Thus the simple filtering and flatfield we applied here was not good enough to completely whiten the noise.

Similarly, one could set the flatfielding to `False` in the map preprocessing (the rho, kappa maps will have to be reloaded so it's best just to restart the kernel). This will not adjust the flux and SNR in highly non-gaussian noise regions, thus more false-positive events are expected.

---

If you are interested in coadding depth-1 maps, you can see the collab notebook [`Pixell_matched_filtering.ipynb`](https://github.com/simonsobs/pixell_tutorials/blob/master/Pixell_matched_filtering.ipynb), also referenced at the beginning of this tutorial.

Coadding multiple detector arrays and even multiple days of observations will reduce low-weight regions and edge effects as more detectors are available to hit the sky pixels.